In [ ]:
!pip install -U deepeval openai pandas pydantic PyPDF2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 844.6/844.6 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.0/472.0 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.9/226.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 1.9 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found ex

In [ ]:
!pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 9.5 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **gpt-4o**

In [ ]:
import json
import glob
import os
import time
import pandas as pd
from google.colab import userdata

from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

os.environ["OPENAI_API_KEY"] = userdata.get('AgentEval')

def load_interview_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Use .get() to safely handle missing keys
    cv_string = json.dumps(data.get('candidate_cv', {}), indent=2)
    combined_input = f"JOB DESCRIPTION:\n{data.get('job_description', '')}\n\nCANDIDATE CV:\n{cv_string}"

    transcript_str = "\n".join([f"{t['role'].upper()}: {t['content']}" for t in data.get('transcript', [])])

    # Extract scorecard if it exists, otherwise return None
    scorecard = data.get('scorecard', None)
    scorecard_str = json.dumps(scorecard, indent=2) if scorecard else None

    return combined_input, transcript_str, scorecard_str

def evaluate_all_interviews(folder_path):
    # 1. Define Custom G-Eval Metrics based on Gold Standards

    prompt_compliance_metric = GEval(
        name="Prompt-Compliance",
        criteria="""Evaluate the interviewer agent's prompt compliance and conversational behavior out of 10.
        CRITICAL GRADING RULE: Evaluate if the interviewer maintains a professional, neutral tone and generally asks one core question per interaction.
        Do NOT penalize the agent for verbosity if it is reasonably providing context, setting up a scenario, or politely acknowledging the candidate's previous answer before asking its next question. As long as the agent does not overwhelm the candidate with multiple disconnected questions in a single turn, and the primary question is clear and well-formed, score it favorably based on clarity and professionalism. Convert to a 0.0-1.0 scale.""",
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        model="gpt-4o"
    )

    semantic_relevance_metric = GEval(
        name="Semantic Relevance",
        criteria="""Evaluate the semantic alignment of the interviewer's questions with the Job Description and Candidate CV out of 10.
        CRITICAL GRADING RULE: Evaluate based on conceptual and thematic relevance. The target role is defined by the provided Job Description.
        Do NOT penalize the interviewer if the candidate's CV lacks the highly specific or advanced experience required by the Job Description. The interviewer must work with the candidate's actual background. Reward the interviewer for adapting its questions to the candidate's general skill set and prior experience, even if they only cover fundamental or related concepts, rather than forcing questions about specialized tools the candidate does not know. Convert to a 0.0-1.0 scale.""",
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model="gpt-4o"
    )

    structural_consistency_metric = GEval(
        name="Structural Consistency",
        criteria="""Evaluate the structural flow and logical progression of the interview out of 10.
        CRITICAL GRADING RULE: Assess whether the transitions between topics are coherent and natural. Focus heavily on the effective use of chat history: reward the interviewer if it asks relevant follow-up questions based on the candidate's previous answers, and penalize it if it abruptly ignores a response or repeats questions.
        Do not penalize the interview if it does not explicitly feature separate 'behavioral' or 'cognitive' stages. As long as the conversational continuity is maintained and the topic progression makes logical sense, score it favorably based on its flow. Convert to a 0.0-1.0 scale.""",
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        model="gpt-4o"
    )

    evaluation_validity_metric = GEval(
        name="Evaluation Validity",
        criteria="""Evaluate the logic of the generated scorecard out of 10.
        CRITICAL GRADING RULE: Your primary focus must be on the scorecard's internal consistency—does the qualitative justification match the final numerical score and PASS/FAIL decision?
        The evaluator is designed to be holistic. Do not penalize the scorecard for reading between the lines, inferring soft skills (like communication or coachability), or extrapolating candidate potential from brief answers. If the scorecard provides a coherent, logical summary that generally reflects the tone of the interview, score it favorably. Convert to a 0.0-1.0 scale.""",
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model="gpt-4o"
    )

    all_results = []
    files = glob.glob(os.path.join(folder_path, "*.json"))

    print(f"Starting evaluation of {len(files)} interviews...\n")

    # 2. Individual Evaluation Loop
    for file_path in files:
        file_name = os.path.basename(file_path)
        print(f"Evaluating: {file_name}...")

        system_input, transcript_output, scorecard_output = load_interview_data(file_path)

        # Test Case A: Evaluating the Interviewer Agent (Transcript)
        interviewer_test_case = LLMTestCase(
            input=system_input,
            actual_output=transcript_output
        )

        prompt_compliance_metric.measure(interviewer_test_case)
        semantic_relevance_metric.measure(interviewer_test_case)
        structural_consistency_metric.measure(interviewer_test_case)

        # Test Case B: Evaluating the Evaluator Agent (Scorecard)
        # We only run this if a scorecard exists in the JSON
        eval_score = None
        eval_reason = "No scorecard provided in JSON."

        if scorecard_output:
            scorecard_test_case = LLMTestCase(
                input=f"{system_input}\n\nTRANSCRIPT:\n{transcript_output}",
                actual_output=scorecard_output
            )
            evaluation_validity_metric.measure(scorecard_test_case)
            eval_score = evaluation_validity_metric.score
            eval_reason = evaluation_validity_metric.reason

        # Log results
        print(f"--- Judgement for {file_name} ---")
        print(f"Prompt Compliance: {prompt_compliance_metric.score}")
        print(f"Semantic Relevance: {semantic_relevance_metric.score}")
        print(f"Structural Consistency: {structural_consistency_metric.score}")
        print(f"Evaluation Validity: {eval_score if eval_score is not None else 'N/A'}")
        print("-" * 40)

        all_results.append({
            "Interview_ID": file_name,
            "Prompt_Compliance_Score": prompt_compliance_metric.score,
            "Prompt_Compliance_Reasoning": prompt_compliance_metric.reason,
            "Semantic_Relevance_Score": semantic_relevance_metric.score,
            "Semantic_Relevance_Reasoning": semantic_relevance_metric.reason,
            "Structural_Consistency_Score": structural_consistency_metric.score,
            "Structural_Consistency_Reasoning": structural_consistency_metric.reason,
            "Eval_Validity_Score": eval_score,
            "Eval_Validity_Reasoning": eval_reason
        })

        time.sleep(25)

    # 3. Full Evaluation (System-Level Metrics)
    df = pd.DataFrame(all_results)

    print("\n========================================")
    print("FULL AGENT EVALUATION METRICS")
    print("========================================")
    print(f"Total Interviews Processed: {len(df)}")
    print(f"Avg Prompt Compliance: {df['Prompt_Compliance_Score'].mean():.2f}")
    print(f"Avg Semantic Relevance: {df['Semantic_Relevance_Score'].mean():.2f}")
    print(f"Avg Structural Consistency: {df['Structural_Consistency_Score'].mean():.2f}")

    # Calculate average Eval Validity only for rows that actually had a scorecard
    valid_scorecards = df['Eval_Validity_Score'].dropna()
    if not valid_scorecards.empty:
        print(f"Avg Evaluation Validity: {valid_scorecards.mean():.2f} ({len(valid_scorecards)} scorecards evaluated)")
    else:
        print("Avg Evaluation Validity: N/A (No scorecards found)")

    # Save the full report to a CSV
    csv_filename = "full_agent_evaluation_report_gpt_4o.csv"
    df.to_csv(csv_filename, index=False)
    print(f"\nDetailed report saved to: {csv_filename}")

# Run the evaluation
if __name__ == "__main__":
    evaluate_all_interviews("/content/drive/MyDrive/Evaluation Dataset")

Starting evaluation of 13 interviews...

Evaluating: Interview Eval #1.json...


Output()

Output()

Output()

--- Judgement for Interview Eval #1.json ---
Prompt Compliance: 0.6522603625994126
Semantic Relevance: 0.24690936462026158
Structural Consistency: 0.41049750934093676
Evaluation Validity: N/A
----------------------------------------
Evaluating: Interview Eval #2.json...


...

Output()

Output()

Output()

Output()

--- Judgement for Interview Eval #12.json ---
Prompt Compliance: 0.5707201359063141
Semantic Relevance: 0.3003580553232796
Structural Consistency: 0.3203371887979162
Evaluation Validity: 0.7714841984088217
----------------------------------------

FULL AGENT EVALUATION METRICS
Total Interviews Processed: 13
Avg Prompt Compliance: 0.67
Avg Semantic Relevance: 0.36
Avg Structural Consistency: 0.52
Avg Evaluation Validity: 0.53 (8 scorecards evaluated)

Detailed report saved to: full_agent_evaluation_report_gpt_4o.csv


# **gpt-5.4-mini-2026-03-17**

In [ ]:
import json
import glob
import os
import time
import pandas as pd
from google.colab import userdata

from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

os.environ["OPENAI_API_KEY"] = userdata.get('AgentEval')

def load_interview_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Use .get() to safely handle missing keys
    cv_string = json.dumps(data.get('candidate_cv', {}), indent=2)
    combined_input = f"JOB DESCRIPTION:\n{data.get('job_description', '')}\n\nCANDIDATE CV:\n{cv_string}"

    transcript_str = "\n".join([f"{t['role'].upper()}: {t['content']}" for t in data.get('transcript', [])])

    # Extract scorecard if it exists, otherwise return None
    scorecard = data.get('scorecard', None)
    scorecard_str = json.dumps(scorecard, indent=2) if scorecard else None

    return combined_input, transcript_str, scorecard_str

def evaluate_all_interviews(folder_path):
    # 1. Define Custom G-Eval Metrics based on Gold Standards

    prompt_compliance_metric = GEval(
        name="Prompt-Compliance",
        criteria="""Evaluate the interviewer agent's prompt compliance and conversational behavior out of 10.
        CRITICAL GRADING RULE: Evaluate if the interviewer maintains a professional, neutral tone and generally asks one core question per interaction.
        Do NOT penalize the agent for verbosity if it is reasonably providing context, setting up a scenario, or politely acknowledging the candidate's previous answer before asking its next question. As long as the agent does not overwhelm the candidate with multiple disconnected questions in a single turn, and the primary question is clear and well-formed, score it favorably based on clarity and professionalism. Convert to a 0.0-1.0 scale.""",
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        model="gpt-5.4-mini-2026-03-17"
    )

    semantic_relevance_metric = GEval(
        name="Semantic Relevance",
        criteria="""Evaluate the semantic alignment of the interviewer's questions with the Job Description and Candidate CV out of 10.
        CRITICAL GRADING RULE: Evaluate based on conceptual and thematic relevance. The target role is defined by the provided Job Description.
        Do NOT penalize the interviewer if the candidate's CV lacks the highly specific or advanced experience required by the Job Description. The interviewer must work with the candidate's actual background. Reward the interviewer for adapting its questions to the candidate's general skill set and prior experience, even if they only cover fundamental or related concepts, rather than forcing questions about specialized tools the candidate does not know. Convert to a 0.0-1.0 scale.""",
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model="gpt-5.4-mini-2026-03-17"
    )

    structural_consistency_metric = GEval(
        name="Structural Consistency",
        criteria="""Evaluate the structural flow and logical progression of the interview out of 10.
        CRITICAL GRADING RULE: Assess whether the transitions between topics are coherent and natural. Focus heavily on the effective use of chat history: reward the interviewer if it asks relevant follow-up questions based on the candidate's previous answers, and penalize it if it abruptly ignores a response or repeats questions.
        Do not penalize the interview if it does not explicitly feature separate 'behavioral' or 'cognitive' stages. As long as the conversational continuity is maintained and the topic progression makes logical sense, score it favorably based on its flow. Convert to a 0.0-1.0 scale.""",
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        model="gpt-5.4-mini-2026-03-17"
    )

    evaluation_validity_metric = GEval(
        name="Evaluation Validity",
        criteria="""Evaluate the logic of the generated scorecard out of 10.
        CRITICAL GRADING RULE: Your primary focus must be on the scorecard's internal consistency—does the qualitative justification match the final numerical score and PASS/FAIL decision?
        The evaluator is designed to be holistic. Do not penalize the scorecard for reading between the lines, inferring soft skills (like communication or coachability), or extrapolating candidate potential from brief answers. If the scorecard provides a coherent, logical summary that generally reflects the tone of the interview, score it favorably. Convert to a 0.0-1.0 scale.""",
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model="gpt-5.4-mini-2026-03-17"
    )


    all_results = []
    files = glob.glob(os.path.join(folder_path, "*.json"))

    print(f"Starting evaluation of {len(files)} interviews...\n")

    # 2. Individual Evaluation Loop
    for file_path in files:
        file_name = os.path.basename(file_path)
        print(f"Evaluating: {file_name}...")

        system_input, transcript_output, scorecard_output = load_interview_data(file_path)

        # Test Case A: Evaluating the Interviewer Agent (Transcript)
        interviewer_test_case = LLMTestCase(
            input=system_input,
            actual_output=transcript_output
        )

        prompt_compliance_metric.measure(interviewer_test_case)
        semantic_relevance_metric.measure(interviewer_test_case)
        structural_consistency_metric.measure(interviewer_test_case)

        # Test Case B: Evaluating the Evaluator Agent (Scorecard)
        # We only run this if a scorecard exists in the JSON
        eval_score = None
        eval_reason = "No scorecard provided in JSON."

        if scorecard_output:
            scorecard_test_case = LLMTestCase(
                input=f"{system_input}\n\nTRANSCRIPT:\n{transcript_output}",
                actual_output=scorecard_output
            )
            evaluation_validity_metric.measure(scorecard_test_case)
            eval_score = evaluation_validity_metric.score
            eval_reason = evaluation_validity_metric.reason

        # Log results
        print(f"--- Judgement for {file_name} ---")
        print(f"Prompt Compliance: {prompt_compliance_metric.score}")
        print(f"Semantic Relevance: {semantic_relevance_metric.score}")
        print(f"Structural Consistency: {structural_consistency_metric.score}")
        print(f"Evaluation Validity: {eval_score if eval_score is not None else 'N/A'}")
        print("-" * 40)

        all_results.append({
            "Interview_ID": file_name,
            "Prompt_Compliance_Score": prompt_compliance_metric.score,
            "Prompt_Compliance_Reasoning": prompt_compliance_metric.reason,
            "Semantic_Relevance_Score": semantic_relevance_metric.score,
            "Semantic_Relevance_Reasoning": semantic_relevance_metric.reason,
            "Structural_Consistency_Score": structural_consistency_metric.score,
            "Structural_Consistency_Reasoning": structural_consistency_metric.reason,
            "Eval_Validity_Score": eval_score,
            "Eval_Validity_Reasoning": eval_reason
        })

        time.sleep(25)

    # 3. Full Evaluation (System-Level Metrics)
    df = pd.DataFrame(all_results)

    print("\n========================================")
    print("FULL AGENT EVALUATION METRICS")
    print("========================================")
    print(f"Total Interviews Processed: {len(df)}")
    print(f"Avg Prompt Compliance: {df['Prompt_Compliance_Score'].mean():.2f}")
    print(f"Avg Semantic Relevance: {df['Semantic_Relevance_Score'].mean():.2f}")
    print(f"Avg Structural Consistency: {df['Structural_Consistency_Score'].mean():.2f}")

    # Calculate average Eval Validity only for rows that actually had a scorecard
    valid_scorecards = df['Eval_Validity_Score'].dropna()
    if not valid_scorecards.empty:
        print(f"Avg Evaluation Validity: {valid_scorecards.mean():.2f} ({len(valid_scorecards)} scorecards evaluated)")
    else:
        print("Avg Evaluation Validity: N/A (No scorecards found)")

    # Save the full report to a CSV
    csv_filename = "full_agent_evaluation_report_gpt_5-4.csv"
    df.to_csv(csv_filename, index=False)
    print(f"\nDetailed report saved to: {csv_filename}")

# Run the evaluation
if __name__ == "__main__":
    evaluate_all_interviews("/content/drive/MyDrive/Evaluation Dataset")

Output()

Starting evaluation of 13 interviews...

Evaluating: Interview Eval #1.json...


Output()

Output()

--- Judgement for Interview Eval #1.json ---
Prompt Compliance: 0.4
Semantic Relevance: 0.2
Structural Consistency: 0.4
Evaluation Validity: N/A
----------------------------------------


...

Evaluating: Interview Eval #12.json...


Output()

Output()

Output()

--- Judgement for Interview Eval #12.json ---
Prompt Compliance: 0.8
Semantic Relevance: 0.2
Structural Consistency: 0.4
Evaluation Validity: 0.8
----------------------------------------

FULL AGENT EVALUATION METRICS
Total Interviews Processed: 13
Avg Prompt Compliance: 0.75
Avg Semantic Relevance: 0.39
Avg Structural Consistency: 0.57
Avg Evaluation Validity: 0.55 (8 scorecards evaluated)

Detailed report saved to: full_agent_evaluation_report_gpt_5-4.csv


# **Gemini**

In [ ]:
import json
import glob
import os
import time
import pandas as pd
from google.colab import userdata

from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import DeepEvalBaseLLM
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = userdata.get('Gemini_API_Key')

class GoogleGemini(DeepEvalBaseLLM):
    def __init__(self, model_name="gemini-2.5-pro"):
        self.model_name = model_name
        self.model = ChatGoogleGenerativeAI(
            model=model_name,
            max_retries=3
        )

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        return self.model.invoke(prompt).content

    async def a_generate(self, prompt: str) -> str:
        res = await self.model.ainvoke(prompt)
        return res.content

    def get_model_name(self):
        return self.model_name

def load_interview_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    cv_string = json.dumps(data.get('candidate_cv', {}), indent=2)
    combined_input = f"JOB DESCRIPTION:\n{data.get('job_description', '')}\n\nCANDIDATE CV:\n{cv_string}"
    transcript_str = "\n".join([f"{t['role'].upper()}: {t['content']}" for t in data.get('transcript', [])])

    scorecard = data.get('scorecard', None)
    scorecard_str = json.dumps(scorecard, indent=2) if scorecard else None

    return combined_input, transcript_str, scorecard_str

def evaluate_all_interviews(folder_path):
    gemini_judge = GoogleGemini("gemini-2.5-pro")

    prompt_compliance_metric = GEval(
        name="Prompt-Compliance",
        criteria="""Evaluate the interviewer agent's prompt compliance and conversational behavior out of 10.
        CRITICAL GRADING RULE: Evaluate if the interviewer maintains a professional, neutral tone and generally asks one core question per interaction.
        Do NOT penalize the agent for verbosity if it is reasonably providing context, setting up a scenario, or politely acknowledging the candidate's previous answer before asking its next question. As long as the agent does not overwhelm the candidate with multiple disconnected questions in a single turn, and the primary question is clear and well-formed, score it favorably based on clarity and professionalism. Convert to a 0.0-1.0 scale.""",
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        model=gemini_judge
    )

    semantic_relevance_metric = GEval(
        name="Semantic Relevance",
        criteria="""Evaluate the semantic alignment of the interviewer's questions with the Job Description and Candidate CV out of 10.
        CRITICAL GRADING RULE: Evaluate based on conceptual and thematic relevance. The target role is defined by the provided Job Description.
        Do NOT penalize the interviewer if the candidate's CV lacks the highly specific or advanced experience required by the Job Description. The interviewer must work with the candidate's actual background. Reward the interviewer for adapting its questions to the candidate's general skill set and prior experience, even if they only cover fundamental or related concepts, rather than forcing questions about specialized tools the candidate does not know. Convert to a 0.0-1.0 scale.""",
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=gemini_judge
    )

    structural_consistency_metric = GEval(
        name="Structural Consistency",
        criteria="""Evaluate the structural flow and logical progression of the interview out of 10.
        CRITICAL GRADING RULE: Assess whether the transitions between topics are coherent and natural. Focus heavily on the effective use of chat history: reward the interviewer if it asks relevant follow-up questions based on the candidate's previous answers, and penalize it if it abruptly ignores a response or repeats questions.
        Do not penalize the interview if it does not explicitly feature separate 'behavioral' or 'cognitive' stages. As long as the conversational continuity is maintained and the topic progression makes logical sense, score it favorably based on its flow. Convert to a 0.0-1.0 scale.""",
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        model=gemini_judge
    )

    evaluation_validity_metric = GEval(
        name="Evaluation Validity",
        criteria="""Evaluate the logic of the generated scorecard out of 10.
        CRITICAL GRADING RULE: Your primary focus must be on the scorecard's internal consistency—does the qualitative justification match the final numerical score and PASS/FAIL decision?
        The evaluator is designed to be holistic. Do not penalize the scorecard for reading between the lines, inferring soft skills (like communication or coachability), or extrapolating candidate potential from brief answers. If the scorecard provides a coherent, logical summary that generally reflects the tone of the interview, score it favorably. Convert to a 0.0-1.0 scale.""",
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=gemini_judge
    )

    all_results = []
    files = glob.glob(os.path.join(folder_path, "*.json"))

    print(f"Starting evaluation of {len(files)} interviews...\n")

    for file_path in files:
        file_name = os.path.basename(file_path)
        print(f"Evaluating: {file_name}...")

        system_input, transcript_output, scorecard_output = load_interview_data(file_path)

        interviewer_test_case = LLMTestCase(
            input=system_input,
            actual_output=transcript_output
        )

        prompt_compliance_metric.measure(interviewer_test_case)
        semantic_relevance_metric.measure(interviewer_test_case)
        structural_consistency_metric.measure(interviewer_test_case)

        eval_score = None
        eval_reason = "No scorecard provided in JSON."

        if scorecard_output:
            scorecard_test_case = LLMTestCase(
                input=f"{system_input}\n\nTRANSCRIPT:\n{transcript_output}",
                actual_output=scorecard_output
            )
            evaluation_validity_metric.measure(scorecard_test_case)
            eval_score = evaluation_validity_metric.score
            eval_reason = evaluation_validity_metric.reason

        print(f"--- Judgement for {file_name} ---")
        print(f"Prompt Compliance: {prompt_compliance_metric.score}")
        print(f"Semantic Relevance: {semantic_relevance_metric.score}")
        print(f"Structural Consistency: {structural_consistency_metric.score}")
        print(f"Evaluation Validity: {eval_score if eval_score is not None else 'N/A'}")
        print("-" * 40)

        all_results.append({
            "Interview_ID": file_name,
            "Prompt_Compliance_Score": prompt_compliance_metric.score,
            "Prompt_Compliance_Reasoning": prompt_compliance_metric.reason,
            "Semantic_Relevance_Score": semantic_relevance_metric.score,
            "Semantic_Relevance_Reasoning": semantic_relevance_metric.reason,
            "Structural_Consistency_Score": structural_consistency_metric.score,
            "Structural_Consistency_Reasoning": structural_consistency_metric.reason,
            "Eval_Validity_Score": eval_score,
            "Eval_Validity_Reasoning": eval_reason
        })

        time.sleep(25)

    df = pd.DataFrame(all_results)

    print("\n========================================")
    print("FULL AGENT EVALUATION METRICS")
    print("========================================")
    print(f"Total Interviews Processed: {len(df)}")
    print(f"Avg Prompt Compliance: {df['Prompt_Compliance_Score'].mean():.2f}")
    print(f"Avg Semantic Relevance: {df['Semantic_Relevance_Score'].mean():.2f}")
    print(f"Avg Structural Consistency: {df['Structural_Consistency_Score'].mean():.2f}")

    valid_scorecards = df['Eval_Validity_Score'].dropna()
    if not valid_scorecards.empty:
        print(f"Avg Evaluation Validity: {valid_scorecards.mean():.2f} ({len(valid_scorecards)} scorecards evaluated)")
    else:
        print("Avg Evaluation Validity: N/A (No scorecards found)")

    csv_filename = "full_agent_evaluation_report_gemini.csv"
    df.to_csv(csv_filename, index=False)
    print(f"\nDetailed report saved to: {csv_filename}")

if __name__ == "__main__":
    evaluate_all_interviews("/content/drive/MyDrive/Evaluation Dataset")

Output()

Starting evaluation of 13 interviews...

Evaluating: Interview Eval #1.json...


Output()

Output()

--- Judgement for Interview Eval #1.json ---
Prompt Compliance: 0.3
Semantic Relevance: 0.1
Structural Consistency: 0.3
Evaluation Validity: N/A
----------------------------------------


...

Evaluating: Interview Eval #12.json...


Output()

Output()

Output()

--- Judgement for Interview Eval #12.json ---
Prompt Compliance: 1.0
Semantic Relevance: 0.1
Structural Consistency: 0.3
Evaluation Validity: 1.0
----------------------------------------

FULL AGENT EVALUATION METRICS
Total Interviews Processed: 13
Avg Prompt Compliance: 0.72
Avg Semantic Relevance: 0.42
Avg Structural Consistency: 0.53
Avg Evaluation Validity: 0.57 (8 scorecards evaluated)

Detailed report saved to: full_agent_evaluation_report_gemini.csv
